# 4차 제출 파일 만들기: 최종 앙상블로 predict_input 927행 예측

| 단계 | 내용 |
|---|---|
| 1 | 4차에서 확정한 방법(`results/final_method.json`): 모델 목록 · 가중치(또는 스태킹 계수) · 임계값 |
| 2 | 테스트 세트 1회 확인: 구성 모델을 개발 6,400행으로 학습 → 테스트 1,600행 예측 → 같은 가중치·임계값으로 결합 |
| 3 | 구성 모델을 8,000행 전체로 재학습 |
| 4 | predict_input 특성 생성 → 모델별 예측 → 결합 → `predict_output_임채현_4차.csv` |

In [1]:
import sys, json
from pathlib import Path
import numpy as np, pandas as pd
HERE = Path.cwd(); M = HERE.parent
sys.path.insert(0, str(M / "common")); sys.path.insert(0, str(M / "exp03_tuning")); sys.path.insert(0, str(HERE))
from preprocess import preprocess_train, preprocess_predict, load_feature_columns
from evaluate import load_splits, predict_score
from submission import write_submission, test_report, INPUT
from load_features import load_all, load_predict, blocks_for
import model_registry as R

final = json.load(open(HERE / "results" / "final_method.json"))
print(json.dumps({k: v for k, v in final.items() if k != "weights"}, ensure_ascii=False, indent=1))
df, X, y, g = preprocess_train(M.parent / "train.csv"); df = load_splits(df); cols = load_feature_columns()
ALL = load_all(df, X); dev = (df.set == "dev").values
specs = {n: R.build(n, cols) for n in final["models"]}
need = sorted({c for _, use in specs.values() for c in use})

def logit(p): p = np.clip(p, 1e-6, 1 - 1e-6); return np.log(p / (1 - p))
def combine(P):
    if final["kind"] == "mean":
        return np.sum([w * P[n] for n, w in zip(final["models"], final["weights"])], 0)
    z = final["intercept"] + np.sum([w * logit(P[n]) for n, w in zip(final["models"], final["weights"])], 0)
    return 1 / (1 + np.exp(-z))

{
 "method": "스태킹: 전체 후보",
 "threshold_mode": "fold 안 임계값",
 "kind": "stack",
 "models": [
  "1차:LogReg",
  "1차:SVM_RBF",
  "1차:KNN_Jaccard",
  "1차:RandomForest",
  "1차:ExtraTrees",
  "1차:LightGBM",
  "1차:XGBoost",
  "1차:CatBoost",
  "1차:MLP",
  "2차:+rdkit217|RandomForest",
  "2차:+rdkit217|LightGBM",
  "2차:+rdkit217|XGBoost",
  "2차:+mordred|RandomForest",
  "2차:+mordred|LightGBM",
  "2차:+mordred|XGBoost",
  "2차:+ptfp|RandomForest",
  "2차:+ptfp|LightGBM",
  "2차:+ptfp|XGBoost",
  "2차:+maccs|RandomForest",
  "2차:+maccs|LightGBM",
  "2차:+maccs|XGBoost",
  "2차:+ap_tt|RandomForest",
  "2차:+ap_tt|LightGBM",
  "2차:+ap_tt|XGBoost",
  "2차:count_chiral|RandomForest",
  "2차:count_chiral|LightGBM",
  "2차:count_chiral|XGBoost",
  "2차:svd|SVM_RBF",
  "2차:svd|MLP",
  "2차:svd|LightGBM",
  "3차:tuned|LightGBM",
  "3차:tuned|XGBoost",
  "3차:tuned|RandomForest"
 ],
 "C": 0.1,
 "threshold": 0.43,
 "cv_f1": 0.8281695588773657,
 "intercept": -0.010870133312503642
}


## 테스트 세트 1회 확인

In [2]:
P = {}
for n, (mk, use) in specs.items():
    m = mk(); m.fit(ALL.loc[dev, use], y[dev]); P[n] = predict_score(m, ALL.loc[~dev, use])
p_test = combine(P)
rep = test_report(y[~dev], p_test, final["threshold"])
print({k: round(v, 4) for k, v in rep.items()})
pd.DataFrame([rep], index=[final["method"]]).to_csv(HERE / "results" / "test_score.csv")

{'F1': 0.8224, 'AUC': 0.8799, 'Precision': 0.778, 'Recall': 0.8721, 'threshold': 0.43}


## 8,000행 전체로 재학습 후 예측

In [3]:
pdf, Xp = preprocess_predict(INPUT)
XP = load_predict(pdf, Xp, blocks_for(need))
P = {}
for n, (mk, use) in specs.items():
    m = mk(); m.fit(ALL[use], y); P[n] = predict_score(m, XP[use])
p = combine(P)
path, sub = write_submission(pdf["SMILES"], (p >= final["threshold"]).astype(int), HERE, "4차")
print("저장:", path.name, "| 행 수", len(sub), "| 예측 비독성(1) 비율", round(sub.iloc[:, 1].mean(), 3), "(train 0.542)")
pd.DataFrame({"SMILES": pdf["SMILES"], "score_nontoxic": p}).to_csv(HERE / "results" / "predict_proba.csv", index=False)
display(sub.head())

저장: predict_output_임채현_4차.csv | 행 수 927 | 예측 비독성(1) 비율 0.599 (train 0.542)


,SMILES,output
0,OC(=O)c1cc2sccc2[nH]1,1
1,[O-][n+]1onc(c2ccccc2)c1C#N,1
2,CN1C(=O)N(C)c2ncn(C)c2C1=O,1
3,Clc1cccc(c1)C2CNCC=CC2,1
4,CCN(CC)CC(=O)Nc1c(C)cccc1C,1
